In [1]:
# Load some test data
import polars as pl
titanic = pl.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/titanic.csv')
titanic.head()

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
i64,str,i64,str,str,f64,i64,i64,str,f64,str,str
1,"""No""",3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S"""
2,"""Yes""",1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C"""
3,"""Yes""",3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S"""
4,"""Yes""",1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S"""
5,"""No""",3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S"""


In [2]:
# Looking ot our survivor ratio, we can see that there are more samples that died than lived
# This example isn't super imbalanced, but it'll serve to illustrate our point
titanic.group_by('Survived').len()

Survived,len
str,u32
"""No""",549
"""Yes""",342


In [3]:
from imblearn.over_sampling import RandomOverSampler
# https://imbalanced-learn.readthedocs.io/en/stable/user_guide.html


# Let's over sample the minority class, which samples with replacement until the
# majority (died) and the minority (survived) are equal
ro = RandomOverSampler()

# Decide which features to use
features = ['Sex', 'Pclass', 'Embarked']
X = titanic.select(features)
y = titanic.select('Survived')

# Oversample, note that we oversample X and y at the same time in order to
# make sure our features and targets stay synched.
X_new, y_new = ro.fit_resample(X.to_numpy(), y.to_numpy())

# Convert this to a dataframe and check the counts, now they're equal, because
# we have a bunch of duplicate survivors
survivors = pl.DataFrame(y_new, schema=['Survived'])
survivors.group_by('Survived').len()

Survived,len
str,u32
"""Yes""",549
"""No""",549


In [4]:
# Another way to accomplish oversampling is to do it natively with Polars
print('Before sampling')
display(titanic.group_by('Survived').len())
number_to_sample = 549 - 342

# 1. Calculate the size difference
# 2. Filters to the just the smaller group
# 3. Sample the smaller group the number of times of the size difference
# 4. Use vstack() to append those new samples to the original data
print('After sampling')
resampled = titanic.filter(pl.col('Survived') == 'Yes').sample(number_to_sample, with_replacement=True).vstack(titanic)
resampled.group_by('Survived').len()

Before sampling


Survived,len
str,u32
"""No""",549
"""Yes""",342


After sampling


Survived,len
str,u32
"""Yes""",549
"""No""",549
